# Advanced Retrieval with LangChain

In the following notebook, we'll explore various methods of advanced retrieval using LangChain!

We'll touch on:

- Naive Retrieval
- Best-Matching 25 (BM25)
- Multi-Query Retrieval
- Parent-Document Retrieval
- Contextual Compression (a.k.a. Rerank)
- Ensemble Retrieval
- Semantic chunking

We'll also discuss how these methods impact performance on our set of documents with a simple RAG chain.

There will be two breakout rooms:

- 🤝 Breakout Room Part #1
  - Task 1: Getting Dependencies!
  - Task 2: Data Collection and Preparation
  - Task 3: Setting Up QDrant!
  - Task 4-10: Retrieval Strategies
- 🤝 Breakout Room Part #2
  - Activity: Evaluate with Ragas

# 🤝 Breakout Room Part #1

## Task 1: Getting Dependencies!

We're going to need a few specific LangChain community packages, like OpenAI (for our [LLM](https://platform.openai.com/docs/models) and [Embedding Model](https://platform.openai.com/docs/guides/embeddings)) and Cohere (for our [Reranker](https://cohere.com/rerank)).

We'll also provide our OpenAI key, as well as our Cohere API key.

In [3]:
import os
import getpass

os.environ["OPENAI_API_KEY"] = getpass.getpass("Enter your OpenAI API Key:")

In [4]:
os.environ["COHERE_API_KEY"] = getpass.getpass("Cohere API Key:")

## Task 2: Data Collection and Preparation

We'll be using our Loan Data once again - this time the strutured data available through the CSV!

### Data Preparation

We want to make sure all our documents have the relevant metadata for the various retrieval strategies we're going to be applying today.

In [5]:
from langchain_community.document_loaders.csv_loader import CSVLoader
from datetime import datetime, timedelta

loader = CSVLoader(
    file_path=f"./data/complaints.csv",
    metadata_columns=[
      "Date received", 
      "Product", 
      "Sub-product", 
      "Issue", 
      "Sub-issue", 
      "Consumer complaint narrative", 
      "Company public response", 
      "Company", 
      "State", 
      "ZIP code", 
      "Tags", 
      "Consumer consent provided?", 
      "Submitted via", 
      "Date sent to company", 
      "Company response to consumer", 
      "Timely response?", 
      "Consumer disputed?", 
      "Complaint ID"
    ]
)

loan_complaint_data = loader.load()

for doc in loan_complaint_data:
    doc.page_content = doc.metadata["Consumer complaint narrative"]

Let's look at an example document to see if everything worked as expected!

In [6]:
loan_complaint_data[0]

Document(metadata={'source': './data/complaints.csv', 'row': 0, 'Date received': '03/27/25', 'Product': 'Student loan', 'Sub-product': 'Federal student loan servicing', 'Issue': 'Dealing with your lender or servicer', 'Sub-issue': 'Trouble with how payments are being handled', 'Consumer complaint narrative': "The federal student loan COVID-19 forbearance program ended in XX/XX/XXXX. However, payments were not re-amortized on my federal student loans currently serviced by Nelnet until very recently. The new payment amount that is effective starting with the XX/XX/XXXX payment will nearly double my payment from {$180.00} per month to {$360.00} per month. I'm fortunate that my current financial position allows me to be able to handle the increased payment amount, but I am sure there are likely many borrowers who are not in the same position. The re-amortization should have occurred once the forbearance ended to reduce the impact to borrowers.", 'Company public response': 'None', 'Company'

## Task 3: Setting up QDrant!

Now that we have our documents, let's create a QDrant VectorStore with the collection name "LoanComplaints".

We'll leverage OpenAI's [`text-embedding-3-small`](https://openai.com/blog/new-embedding-models-and-api-updates) because it's a very powerful (and low-cost) embedding model.

> NOTE: We'll be creating additional vectorstores where necessary, but this pattern is still extremely useful.

In [7]:
from langchain_community.vectorstores import Qdrant
from langchain_openai import OpenAIEmbeddings

embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

vectorstore = Qdrant.from_documents(
    loan_complaint_data,
    embeddings,
    location=":memory:",
    collection_name="LoanComplaints"
)

## Task 4: Naive RAG Chain

Since we're focusing on the "R" in RAG today - we'll create our Retriever first.

### R - Retrieval

This naive retriever will simply look at each review as a document, and use cosine-similarity to fetch the 10 most relevant documents.

> NOTE: We're choosing `10` as our `k` here to provide enough documents for our reranking process later

In [8]:
naive_retriever = vectorstore.as_retriever(search_kwargs={"k" : 10})

### A - Augmented

We're going to go with a standard prompt for our simple RAG chain today! Nothing fancy here, we want this to mostly be about the Retrieval process.

In [9]:
from langchain_core.prompts import ChatPromptTemplate

RAG_TEMPLATE = """\
You are a helpful and kind assistant. Use the context provided below to answer the question.

If you do not know the answer, or are unsure, say you don't know.

Query:
{question}

Context:
{context}
"""

rag_prompt = ChatPromptTemplate.from_template(RAG_TEMPLATE)

### G - Generation

We're going to leverage `gpt-4.1-nano` as our LLM today, as - again - we want this to largely be about the Retrieval process.

In [10]:
from langchain_openai import ChatOpenAI

chat_model = ChatOpenAI(model="gpt-4.1-nano")

### LCEL RAG Chain

We're going to use LCEL to construct our chain.

> NOTE: This chain will be exactly the same across the various examples with the exception of our Retriever!

In [11]:
from langchain_core.runnables import RunnablePassthrough
from operator import itemgetter
from langchain_core.output_parsers import StrOutputParser

naive_retrieval_chain = (
    # INVOKE CHAIN WITH: {"question" : "<<SOME USER QUESTION>>"}
    # "question" : populated by getting the value of the "question" key
    # "context"  : populated by getting the value of the "question" key and chaining it into the base_retriever
    {"context": itemgetter("question") | naive_retriever, "question": itemgetter("question")}
    # "context"  : is assigned to a RunnablePassthrough object (will not be called or considered in the next step)
    #              by getting the value of the "context" key from the previous step
    | RunnablePassthrough.assign(context=itemgetter("context"))
    # "response" : the "context" and "question" values are used to format our prompt object and then piped
    #              into the LLM and stored in a key called "response"
    # "context"  : populated by getting the value of the "context" key from the previous step
    | {"response": rag_prompt | chat_model, "context": itemgetter("context")}
)

Let's see how this simple chain does on a few different prompts.

> NOTE: You might think that we've cherry picked prompts that showcase the individual skill of each of the retrieval strategies - you'd be correct!

In [12]:
naive_retrieval_chain.invoke({"question" : "What is the most common issue with loans?"})["response"].content

"Based on the provided context, the most common issues with student loans include:\n\n- Problems with lender or servicer dealing, such as receiving bad information about the loan, errors in loan balances, misapplied payments, and wrongful denials of payment plans.\n- Incorrect or inconsistent information on credit reports, including false delinquencies and balances.\n- Troubles with payment handling, such as restrictions on applying extra payments to principal or paying off loans early, which can be perceived as predatory.\n- Discrepancies and mishandling related to loan transfers, improper documentation, or unauthorized sharing of personal information.\n- Struggles with repayment plans, including being directed into forbearance repeatedly, interest capitalization, and complex or misleading billing.\n- Issues linked to loan discharge, cancellation, or discharge eligibility, often compounded by long-term forbearance or mismanagement.\n  \nWhile these issues vary, the most prevalent conc

In [13]:
naive_retrieval_chain.invoke({"question" : "Did any complaints not get handled in a timely manner?"})["response"].content

'Based on the provided information, yes, some complaints did not get handled in a timely manner. Specifically, the complaint with Company MOHELA received a "No" response for the "Timely response?" metric, indicating it was not addressed promptly. Additionally, multiple complaints note extended periods of waiting, lack of response, or unresolved issues over several weeks or months.'

In [14]:
naive_retrieval_chain.invoke({"question" : "Why did people fail to pay back their loans?"})["response"].content

"People failed to pay back their loans primarily due to several interconnected reasons highlighted in the complaints:\n\n1. **Accumulating Interest and Financial Hardships**: Many borrowers found that interest continued to accrue even during forbearance or deferment periods, making it difficult to reduce the principal amount. Lowering payments often resulted in higher overall debt due to interest accumulation, extending the repayment period and increasing total costs. Borrowers also faced financial hardships like stagnant wages, unemployment, or unexpected expenses, which made it impossible to increase payments or settle their debt.\n\n2. **Lack of Clear Communication and Unclear Instructions**: Several complaints indicate that borrowers were not adequately informed about their repayment status, due dates, loan transfers, or changes in servicers. For example, some borrowers were unaware when their loans were transferred between companies or when payments resumed, leading to missed paym

Overall, this is not bad! Let's see if we can make it better!

## Task 5: Best-Matching 25 (BM25) Retriever

Taking a step back in time - [BM25](https://www.nowpublishers.com/article/Details/INR-019) is based on [Bag-Of-Words](https://en.wikipedia.org/wiki/Bag-of-words_model) which is a sparse representation of text.

In essence, it's a way to compare how similar two pieces of text are based on the words they both contain.

This retriever is very straightforward to set-up! Let's see it happen down below!


In [15]:
from langchain_community.retrievers import BM25Retriever

bm25_retriever = BM25Retriever.from_documents(loan_complaint_data, )

We'll construct the same chain - only changing the retriever.

In [16]:
bm25_retrieval_chain = (
    {"context": itemgetter("question") | bm25_retriever, "question": itemgetter("question")}
    | RunnablePassthrough.assign(context=itemgetter("context"))
    | {"response": rag_prompt | chat_model, "context": itemgetter("context")}
)

Let's look at the responses!

In [17]:
bm25_retrieval_chain.invoke({"question" : "What is the most common issue with loans?"})["response"].content

'Based on the provided context, a common issue with loans, specifically federal student loans, appears to be dealing with the lender or servicer regarding various problems. The most frequent issues include:\n\n- Disputes over fees charged or incorrect information provided about the loan.\n- Trouble with how payments are being handled, such as restrictions on applying extra funds to the principal or paying off loans more quickly.\n- Receiving bad or confusing information about loan balances, interest, or loan terms.\n- Lack of transparency or trustworthiness of the loan servicers, with accusations of dishonesty or predatory practices.\n\nTherefore, the most common issue with loans, as reflected in these complaints, is **problems related to dealing with the loan lender or servicer, including issues with fees, payment application, and inaccurate information**.'

In [18]:
bm25_retrieval_chain.invoke({"question" : "Did any complaints not get handled in a timely manner?"})["response"].content

'Based on the provided information, all the complaints mentioned were responded to with a "Closed with explanation" status and were marked as "Timely response?": "Yes." Therefore, no complaints appear to have gone unhandled in a timely manner.'

In [19]:
bm25_retrieval_chain.invoke({"question" : "Why did people fail to pay back their loans?"})["response"].content

'People failed to pay back their loans for various reasons, including issues with payment plans, lack of communication from lenders or servicers, and problems with their accounts. Specifically, some borrowers experienced difficulties due to being steered into incorrect types of forbearances or having their autopayments discontinued without proper notification. Others encountered billing problems, such as payments being reversed or not processed correctly, and were not informed about account status changes or transfer of their loans to different servicers. Additionally, some borrowers felt deceived or neglected, believing they were unfairly billed or their repayment options were mismanaged, leading to negative impacts on their credit scores.'

It's not clear that this is better or worse, if only we had a way to test this (SPOILERS: We do, the second half of the notebook will cover this)

#### ❓ Question #1:

Give an example query where BM25 is better than embeddings and justify your answer.

##### ✅ Answer:
BM25 would excel at exact keyword matching tasks. So for instance if I were to ask "What is the complaint ID for the issue about 'incorrect interest rate'?", it would be able to find that exact phrase in a user's complaint and match it perfectly. The only extra step here would be to find the complaint ID. Embeddings are much better for questions that require semantic understanding of contexts and data.

## Task 6: Contextual Compression (Using Reranking)

Contextual Compression is a fairly straightforward idea: We want to "compress" our retrieved context into just the most useful bits.

There are a few ways we can achieve this - but we're going to look at a specific example called reranking.

The basic idea here is this:

- We retrieve lots of documents that are very likely related to our query vector
- We "compress" those documents into a smaller set of *more* related documents using a reranking algorithm.

We'll be leveraging Cohere's Rerank model for our reranker today!

All we need to do is the following:

- Create a basic retriever
- Create a compressor (reranker, in this case)

That's it!

Let's see it in the code below!

In [20]:
from langchain.retrievers.contextual_compression import ContextualCompressionRetriever
from langchain_cohere import CohereRerank

compressor = CohereRerank(model="rerank-v3.5")
compression_retriever = ContextualCompressionRetriever(
    base_compressor=compressor, base_retriever=naive_retriever
)

Let's create our chain again, and see how this does!

In [21]:
contextual_compression_retrieval_chain = (
    {"context": itemgetter("question") | compression_retriever, "question": itemgetter("question")}
    | RunnablePassthrough.assign(context=itemgetter("context"))
    | {"response": rag_prompt | chat_model, "context": itemgetter("context")}
)

In [22]:
contextual_compression_retrieval_chain.invoke({"question" : "What is the most common issue with loans?"})["response"].content

'Based on the provided context, the most common issue with loans appears to be problems related to dealing with lenders or servicers. Specific issues include incorrect or bad information about the loans, errors in loan balances, misapplied payments, wrongful denials of payment plans, and mishandling of loan data. Many complaints also involve lack of communication, inaccurate record-keeping, and violations of privacy laws.'

In [23]:
contextual_compression_retrieval_chain.invoke({"question" : "Did any complaints not get handled in a timely manner?"})["response"].content

'Based on the provided complaints, one complaint indicates that it has been nearly 18 months with no resolution, which suggests it was not handled in a timely manner. The other complaints, despite some issues, are noted to have been responded to in a timely manner, with responses marked as "Yes."'

In [24]:
contextual_compression_retrieval_chain.invoke({"question" : "Why did people fail to pay back their loans?"})["response"].content

'People failed to pay back their loans for several reasons, including:\n\n1. Lack of Awareness and Information: Some borrowers were not informed about their obligation to repay student loans, especially if they were the first in their families to attend college and did not receive proper guidance from financial aid officers.\n\n2. Compounding Interest and Unaffordable Payments: Borrowers often found that interest continued to accrue even when payments were deferred or when they tried to lower monthly payments, making it difficult to pay down the principal and leading to balances that grew over time.\n\n3. Financial Hardship and Unmanageable Debt: Many borrowers experienced financial hardships, making it hard to keep up with payments while maintaining basic living expenses. For example, increased balances despite payments, or being unable to increase payments due to financial constraints.\n\n4. Issues with Loan Servicers and Communication: Problems such as not being notified of account 

We'll need to rely on something like Ragas to help us get a better sense of how this is performing overall - but it "feels" better!

## Task 7: Multi-Query Retriever

Typically in RAG we have a single query - the one provided by the user.

What if we had....more than one query!

In essence, a Multi-Query Retriever works by:

1. Taking the original user query and creating `n` number of new user queries using an LLM.
2. Retrieving documents for each query.
3. Using all unique retrieved documents as context

So, how is it to set-up? Not bad! Let's see it down below!



In [25]:
from langchain.retrievers.multi_query import MultiQueryRetriever

multi_query_retriever = MultiQueryRetriever.from_llm(
    retriever=naive_retriever, llm=chat_model
)

In [26]:
multi_query_retrieval_chain = (
    {"context": itemgetter("question") | multi_query_retriever, "question": itemgetter("question")}
    | RunnablePassthrough.assign(context=itemgetter("context"))
    | {"response": rag_prompt | chat_model, "context": itemgetter("context")}
)

In [27]:
multi_query_retrieval_chain.invoke({"question" : "What is the most common issue with loans?"})["response"].content

'Based on the provided data, the most common issue with loans appears to be problems related to the handling and servicing of student loans, including issues such as:\n\n- Dealing with lenders or servicers (e.g., incorrect or bad information, misapplication of payments, wrongful denials, or improper transfer of loans)\n- Struggles with repayment plans and payment handling (e.g., interest accumulation, inability to pay on time, problems with forbearance or deferment)\n- Disputes over loan validity, transfer, and legality due to agency restructuring or administrative changes\n- Errors in loan balances, misapplied payments, and inaccurate reporting that affects credit scores\n- Lack of proper documentation and verification, including missing or unverified Master Promissory Notes (MPNs)\n- Issues with income-driven repayment plans and loan forgiveness programs\n\nOverall, many complaints center around poor communication, illegal or improper servicing practices, and administrative errors. I

In [28]:
multi_query_retrieval_chain.invoke({"question" : "Did any complaints not get handled in a timely manner?"})["response"].content

'Yes, based on the complaints reviewed, some complaints were not handled in a timely manner. Specifically, there are multiple instances where responses were marked as "No" for timely response or the consumer reported delays exceeding the expected timeframe. For example, complaints filed with responses marked as "No" for timely response include those received on 03/28/25 and 04/01/25 from Mohela, and others from EdFinancial Services, indicating delays beyond the expected response periods.'

In [29]:
multi_query_retrieval_chain.invoke({"question" : "Why did people fail to pay back their loans?"})["response"].content

'People failed to pay back their loans for various reasons, including:\n\n- Errors and misconduct by loan servicers, such as misapplied payments, wrongful denials of payment plans, and errors in loan balances.\n- Poor communication or lack of proper notices from lenders about repayment obligations or changes in loan status.\n- Illegal or misleading practices like forbearance steering, where borrowers are pushed into forbearance without being informed of alternative, more manageable repayment options like income-driven plans or rehabilitation.\n- Transfer of loans between companies without proper notification, leading to confusion and missed payments.\n- Financial hardships stemming from unexpected life events such as homelessness, accidents, or unemployment, which make repayment difficult.\n- Loan management issues, including incorrect reporting of delinquency or defaults, and administrative errors.\n- Lack of access to resources or support to understand or navigate repayment options, 

#### ❓ Question #2:

Explain how generating multiple reformulations of a user query can improve recall.

##### ✅ Answer:
It increases the chances of retrieving all of the relevant documents, even if different wording/phrasing is used than that of the original query. This reduces the risk of missing relevant info due to varying verbage.

## Task 8: Parent Document Retriever

A "small-to-big" strategy - the Parent Document Retriever works based on a simple strategy:

1. Each un-split "document" will be designated as a "parent document" (You could use larger chunks of document as well, but our data format allows us to consider the overall document as the parent chunk)
2. Store those "parent documents" in a memory store (not a VectorStore)
3. We will chunk each of those documents into smaller documents, and associate them with their respective parents, and store those in a VectorStore. We'll call those "child chunks".
4. When we query our Retriever, we will do a similarity search comparing our query vector to the "child chunks".
5. Instead of returning the "child chunks", we'll return their associated "parent chunks".

Okay, maybe that was a few steps - but the basic idea is this:

- Search for small documents
- Return big documents

The intuition is that we're likely to find the most relevant information by limiting the amount of semantic information that is encoded in each embedding vector - but we're likely to miss relevant surrounding context if we only use that information.

Let's start by creating our "parent documents" and defining a `RecursiveCharacterTextSplitter`.

In [30]:
from langchain.retrievers import ParentDocumentRetriever
from langchain.storage import InMemoryStore
from langchain_text_splitters import RecursiveCharacterTextSplitter
from qdrant_client import QdrantClient, models

parent_docs = loan_complaint_data
child_splitter = RecursiveCharacterTextSplitter(chunk_size=750)

We'll need to set up a new QDrant vectorstore - and we'll use another useful pattern to do so!

> NOTE: We are manually defining our embedding dimension, you'll need to change this if you're using a different embedding model.

In [31]:
from langchain_qdrant import QdrantVectorStore

client = QdrantClient(location=":memory:")

client.create_collection(
    collection_name="full_documents",
    vectors_config=models.VectorParams(size=1536, distance=models.Distance.COSINE)
)

parent_document_vectorstore = QdrantVectorStore(
    collection_name="full_documents", embedding=OpenAIEmbeddings(model="text-embedding-3-small"), client=client
)

Now we can create our `InMemoryStore` that will hold our "parent documents" - and build our retriever!

In [32]:
store = InMemoryStore()

parent_document_retriever = ParentDocumentRetriever(
    vectorstore = parent_document_vectorstore,
    docstore=store,
    child_splitter=child_splitter,
)

By default, this is empty as we haven't added any documents - let's add some now!

In [33]:
parent_document_retriever.add_documents(parent_docs, ids=None)

We'll create the same chain we did before - but substitute our new `parent_document_retriever`.

In [34]:
parent_document_retrieval_chain = (
    {"context": itemgetter("question") | parent_document_retriever, "question": itemgetter("question")}
    | RunnablePassthrough.assign(context=itemgetter("context"))
    | {"response": rag_prompt | chat_model, "context": itemgetter("context")}
)

Let's give it a whirl!

In [35]:
parent_document_retrieval_chain.invoke({"question" : "What is the most common issue with loans?"})["response"].content

'The most common issue with loans, based on the complaints provided, appears to be related to problems with servicing and reporting, including errors in loan balances, misapplied payments, wrongful denials of payment plans, and issues arising from loan transfers and reporting to credit bureaus. Additionally, concerns about unfair and deceptive practices, such as unexpected interest rate increases and incorrect information on credit reports, are frequently mentioned.'

In [36]:
parent_document_retrieval_chain.invoke({"question" : "Did any complaints not get handled in a timely manner?"})["response"].content

'Based on the provided information, several complaints were marked as "No" for being handled in a timely manner. Specifically, at least two complaints regarding federal student loan servicing by MOHELA (Complaint ID: 12709087 and 12935889) involved delays in response, with the narrative indicating that the complainants had not heard back as of the dates of their complaints. The first complaint explicitly states the complainant "has not heard from anyone" despite being told a response would take 15 days. The second complaint similarly indicates ongoing issues with delays.\n\nAdditionally, these complaints were closed with explanations but were explicitly noted as "No" for timely response, indicating they did not receive a prompt or timely handling.\n\nTherefore, yes, there were complaints that were not handled in a timely manner.'

In [37]:
parent_document_retrieval_chain.invoke({"question" : "Why did people fail to pay back their loans?"})["response"].content

'People often failed to pay back their loans due to a variety of issues, including mismanagement by loan servicers, lack of proper notification about payment obligations, insufficient information about repayment processes, and financial hardships. For example, some borrowers experienced payments being resumed prematurely while still attending school or after the completion of their studies without proper reevaluation of their repayment terms. Others faced difficulties because the institutions they attended provided misleading information about the value of their degrees and the job prospects, which contributed to their inability to secure employment and repay loans. Additionally, issues such as administrative errors, failure to verify the legitimacy of debts, and lack of clear communication about account changes also hindered borrowers from making timely payments. Overall, these factors contributed to borrowers struggling to meet their repayment obligations.'

Overall, the performance *seems* largely the same. We can leverage a tool like [Ragas]() to more effectively answer the question about the performance.

## Task 9: Ensemble Retriever

In brief, an Ensemble Retriever simply takes 2, or more, retrievers and combines their retrieved documents based on a rank-fusion algorithm.

In this case - we're using the [Reciprocal Rank Fusion](https://plg.uwaterloo.ca/~gvcormac/cormacksigir09-rrf.pdf) algorithm.

Setting it up is as easy as providing a list of our desired retrievers - and the weights for each retriever.

In [38]:
from langchain.retrievers import EnsembleRetriever

retriever_list = [bm25_retriever, naive_retriever, parent_document_retriever, compression_retriever, multi_query_retriever]
equal_weighting = [1/len(retriever_list)] * len(retriever_list)

ensemble_retriever = EnsembleRetriever(
    retrievers=retriever_list, weights=equal_weighting
)

We'll pack *all* of these retrievers together in an ensemble.

In [39]:
ensemble_retrieval_chain = (
    {"context": itemgetter("question") | ensemble_retriever, "question": itemgetter("question")}
    | RunnablePassthrough.assign(context=itemgetter("context"))
    | {"response": rag_prompt | chat_model, "context": itemgetter("context")}
)

Let's look at our results!

In [40]:
ensemble_retrieval_chain.invoke({"question" : "What is the most common issue with loans?"})["response"].content

'The most common issue with loans, based on the provided data, appears to be problems related to dealing with lenders or servicers, such as receiving inaccurate or bad information about loans, misapplied payments, unauthorized transfers, and lack of proper communication and notification. Many complaints highlight errors in loan balances, interest calculations, reporting to credit bureaus, and difficulties in obtaining clear and transparent information about loan terms and status.'

In [41]:
ensemble_retrieval_chain.invoke({"question" : "Did any complaints not get handled in a timely manner?"})["response"].content

'Based on the provided complaints, yes, some complaints were not handled in a timely manner. Specifically, there is at least one record indicating a "No" response to whether the response was timely, such as:\n\n- Complaint ID: 12739706 (MOHELA in NJ) received on 04/01/25, where the response was "Closed with explanation" and "Timely response?": "No".\n\nSimilarly, some other complaints show delays or waiting times exceeding acceptable periods, for example:\n\n- Complaint ID: 12744910 (Maximus Federal Services in MI) received on 03/31/25, where the response was "Closed with explanation" and "Timely response?": "Yes", but the complaint details mention waits of hours and lack of response for over 2-3 weeks.\n\nOverall, the data indicates that at least some complaints did not get handled promptly, either due to delays, waiting times, or responses not received within the expected time frames.'

In [42]:
ensemble_retrieval_chain.invoke({"question" : "Why did people fail to pay back their loans?"})["response"].content

"People failed to pay back their loans for various reasons, as indicated in the complaints:\n\n1. **Lack of Clear Information and Miscommunication:** Many borrowers were unaware of their repayment obligations, loan transfer details, or changes in their loan status due to insufficient or poor communication from servicers. For example, some were not notified when their loans were transferred between companies or when payment resumption was scheduled, leading to unintentional delinquency.\n\n2. **Unfavorable Loan Terms and Rising Interest:** Borrowers reported that interest continued to accrue and compound, making it difficult to reduce their principal balance. Long-term forbearance and lack of transparent options exacerbated this issue, with some borrowers seeing their balances grow despite ongoing payments.\n\n3. **Difficulty Applying Payments Correctly:** Several complaints highlight that making extra payments, applying funds to principal, or paying off smaller loans was obstructed by 

## Task 10: Semantic Chunking

While this is not a retrieval method - it *is* an effective way of increasing retrieval performance on corpora that have clean semantic breaks in them.

Essentially, Semantic Chunking is implemented by:

1. Embedding all sentences in the corpus.
2. Combining or splitting sequences of sentences based on their semantic similarity based on a number of [possible thresholding methods](https://python.langchain.com/docs/how_to/semantic-chunker/):
  - `percentile`
  - `standard_deviation`
  - `interquartile`
  - `gradient`
3. Each sequence of related sentences is kept as a document!

Let's see how to implement this!

We'll use the `percentile` thresholding method for this example which will:

Calculate all distances between sentences, and then break apart sequences of setences that exceed a given percentile among all distances.

In [43]:
from langchain_experimental.text_splitter import SemanticChunker

semantic_chunker = SemanticChunker(
    embeddings,
    breakpoint_threshold_type="percentile"
)

Now we can split our documents.

In [44]:
semantic_documents = semantic_chunker.split_documents(loan_complaint_data[:20])

Let's create a new vector store.

In [45]:
semantic_vectorstore = Qdrant.from_documents(
    semantic_documents,
    embeddings,
    location=":memory:",
    collection_name="Loan_Complaint_Data_Semantic_Chunks"
)

We'll use naive retrieval for this example.

In [46]:
semantic_retriever = semantic_vectorstore.as_retriever(search_kwargs={"k" : 10})

Finally we can create our classic chain!

In [47]:
semantic_retrieval_chain = (
    {"context": itemgetter("question") | semantic_retriever, "question": itemgetter("question")}
    | RunnablePassthrough.assign(context=itemgetter("context"))
    | {"response": rag_prompt | chat_model, "context": itemgetter("context")}
)

And view the results!

In [48]:
semantic_retrieval_chain.invoke({"question" : "What is the most common issue with loans?"})["response"].content

'The most common issue with loans, based on the provided complaints, appears to be problems related to loan servicing and management. This includes issues such as difficulty in obtaining accurate and timely information about loan status, miscommunication or lack of communication from lenders or servicers, difficulties with repayment plans (e.g., incorrect payment amounts, problems with auto-debit setups), and improper reporting or handling of loan status (such as loans being mistakenly reported as in default or delinquent). Many complaints also highlight concerns about the mishandling of borrower data and violations of privacy laws.\n\nIn summary, the predominant issues involve poor communication, errors in loan management, and mishandling of borrower information within the loan servicing process.'

In [49]:
semantic_retrieval_chain.invoke({"question" : "Did any complaints not get handled in a timely manner?"})["response"].content

'Based on the provided complaints, several complaints indicate issues with timely handling or response:\n\n- Complaint 17 (05/04/25, Nelnet): Despite multiple letters sent via Certified Mail, Nelnet never responded to the complaint or provided answers, indicating a lack of timely handling.\n- Complaint 14 (04/28/25, Maximus): Response was "Closed with explanation," but the complaint does not specify whether the response was timely, so we cannot conclusively say if it was late.\n- Complaint 4 (05/05/25, MOHELA): Response was "Closed with explanation," but the timeliness is marked "Yes," suggesting it was handled promptly.\n- Complaint 1 (05/09/25, Maximus): Response classified as "Closed with explanation" and marked "Yes" for timely response.\n- Complaints regarding disputes and instructions for investigations (e.g., complaints 19, 11, 16, etc.) indicate ongoing or unresolved issues, but specific delays or failures to respond in a timely manner are not explicitly detailed.\n\n**Summary:

In [50]:
semantic_retrieval_chain.invoke({"question" : "Why did people fail to pay back their loans?"})["response"].content

"People failed to pay back their loans for various reasons, including challenges with loan management and issues related to the loan servicing process. Some specific issues mentioned in the complaints include:\n\n- Receiving incorrect or bad information about their loans, which led to misunderstandings about their loan status or repayment obligations.\n- Difficulties in contacting or getting clear responses from loan servicers, leading to frustration and delays in resolving issues.\n- Problems with re-amortization and payment processing, resulting in unexpected increases in monthly payments or payments not being credited correctly.\n- Disputes over loan legitimacy, such as loans being reported as in default despite the borrowers’ claims of never defaulting.\n- Complications arising from loan transfers and missing payments, causing confusion and potential default.\n- Challenges with loan forgiveness programs or discharge requests, where delays or mismanagement hindered borrowers' effort

#### ❓ Question #3:

If sentences are short and highly repetitive (e.g., FAQs), how might semantic chunking behave, and how would you adjust the algorithm?

##### ✅ Answer:
Performance would definitely suffer for semantic chunking, due to low semantic variation. This would also result in over chunking, creating too many tiny chunks since no clear semantic boundaries can be found. I would adjust the algorithm by increasing the minimum chunk size and using different thresholding other than percentile. Pre-processing to remove exact duplicates before chunking would also remediate many of the problems with FAQ style sentences.

# 🤝 Breakout Room Part #2

#### 🏗️ Activity #1

Your task is to evaluate the various Retriever methods against eachother.

You are expected to:

1. Create a "golden dataset"
 - Use Synthetic Data Generation (powered by Ragas, or otherwise) to create this dataset
2. Evaluate each retriever with *retriever specific* Ragas metrics
 - Semantic Chunking is not considered a retriever method and will not be required for marks, but you may find it useful to do a "semantic chunking on" vs. "semantic chunking off" comparision between them
3. Compile these in a list and write a small paragraph about which is best for this particular data and why.

Your analysis should factor in:
  - Cost
  - Latency
  - Performance

> NOTE: This is **NOT** required to be completed in class. Please spend time in your breakout rooms creating a plan before moving on to writing code.

##### HINTS:

- LangSmith provides detailed information about latency and cost.

In [55]:
from langchain_community.document_loaders import DirectoryLoader
from langchain_community.document_loaders import PyMuPDFLoader
from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper
from langchain_openai import ChatOpenAI
from langchain_openai import OpenAIEmbeddings
from ragas.testset import TestsetGenerator

path = "data/"
loader = DirectoryLoader(path, glob="*.pdf", loader_cls=PyMuPDFLoader)
docs = loader.load()

generator_llm = LangchainLLMWrapper(ChatOpenAI(model="gpt-4.1"))
generator_embeddings = LangchainEmbeddingsWrapper(OpenAIEmbeddings())


generator = TestsetGenerator(llm=generator_llm, embedding_model=generator_embeddings)
dataset = generator.generate_with_langchain_docs(docs[:20], testset_size=10)

dataset.to_pandas()



Applying HeadlinesExtractor:   0%|          | 0/17 [00:00<?, ?it/s]

Applying HeadlineSplitter:   0%|          | 0/20 [00:00<?, ?it/s]

unable to apply transformation: 'headlines' property not found in this node
unable to apply transformation: 'headlines' property not found in this node
unable to apply transformation: 'headlines' property not found in this node


Applying SummaryExtractor:   0%|          | 0/29 [00:00<?, ?it/s]

Property 'summary' already exists in node '538893'. Skipping!
Property 'summary' already exists in node '830917'. Skipping!
Property 'summary' already exists in node '6d7d17'. Skipping!
Property 'summary' already exists in node 'ea45ef'. Skipping!
Property 'summary' already exists in node 'fc108f'. Skipping!
Property 'summary' already exists in node '60c7c0'. Skipping!
Property 'summary' already exists in node 'f72b59'. Skipping!
Property 'summary' already exists in node 'a44b78'. Skipping!
Property 'summary' already exists in node '8a5bdd'. Skipping!
Property 'summary' already exists in node '9acb83'. Skipping!
Property 'summary' already exists in node '1fd373'. Skipping!
Property 'summary' already exists in node '437f1b'. Skipping!


...

Applying [CosineSimilarityBuilder, OverlapScoreBuilder]:   0%|          | 0/2 [00:00<?, ?it/s]

Generating personas:   0%|          | 0/3 [00:00<?, ?it/s]

Generating Scenarios:   0%|          | 0/3 [00:00<?, ?it/s]

Generating Samples:   0%|          | 0/12 [00:00<?, ?it/s]

,user_input,reference_contexts,reference,synthesizer_name
0,if student parent dont wanna fill out FAFSA wh...,"[information, see the discussion under <Direct...",If you verify that the parents of a dependent ...,single_hop_specifc_query_synthesizer
1,Whoo can fill out the FASFA for a Direct PLUS ...,[Definition of <Parent= for Direct PLUS Loan P...,"The student’s biological parent, legal adoptiv...",single_hop_specifc_query_synthesizer
2,How does the Department use the NSLDS to check...,[If your school participates in the Direct PLU...,The Department conducts an NSLDS default check...,single_hop_specifc_query_synthesizer
3,"According to DCL GEN-16-10, what are the compl...",[Direct Loan Eligibility After an Enrollment S...,DCL GEN-16-10 specifies that if a student who ...,single_hop_specifc_query_synthesizer
4,Who can get Direct PLUS Loans for a dependent ...,"[<1-hop>\n\ninformation, see the discussion un...",Direct PLUS Loans can be borrowed by the biolo...,multi_hop_abstract_query_synthesizer
5,If a school chooses to participate in the Dire...,"[<1-hop>\n\ninformation, see the discussion un...",If a school participates in the Direct PLUS Lo...,multi_hop_abstract_query_synthesizer
6,If a dependent undergraduate student's parents...,"[<1-hop>\n\ninformation, see the discussion un...",If a dependent undergraduate student's parents...,multi_hop_abstract_query_synthesizer
7,"How does Direct Loan counseling, specifically ...",[<1-hop>\n\nChapter 2 Direct Loan Counseling C...,"Direct Loan counseling, particularly entrance ...",multi_hop_abstract_query_synthesizer
8,How does the Department ensure that students a...,[<1-hop>\n\nprovided by those companies are al...,The Department ensures that students and paren...,multi_hop_specific_query_synthesizer
9,"According to federal regulations, what are the...",[<1-hop>\n\nChapter 2 Direct Loan Counseling C...,Federal regulations require that first-time st...,multi_hop_specific_query_synthesizer


In [ ]:
import copy
from ragas import EvaluationDataset
from ragas import evaluate
from ragas.llms import LangchainLLMWrapper
from ragas.metrics import LLMContextRecall, Faithfulness, FactualCorrectness, ResponseRelevancy, ContextEntityRecall, NoiseSensitivity

retrievers = {
    "naive": naive_retrieval_chain,
    "bm25": bm25_retrieval_chain,
    "compression": contextual_compression_retrieval_chain,
    "multi_query": multi_query_retrieval_chain,
    "parent_document": parent_document_retrieval_chain,
    "ensemble": ensemble_retrieval_chain
}
metric_map = {}
for retriever_name in retrievers:
    copy_dataset = copy.deepcopy(dataset)
    for test_row in dataset:
        retriever_chain = retrievers[retriever_name]
        response = retriever_chain.invoke({"question" : test_row.eval_sample.user_input})
        test_row.eval_sample.response = response["response"]
        test_row.eval_sample.retrieved_contexts = [context.page_content for context in response["context"]]

    evaluator_llm = LangchainLLMWrapper(ChatOpenAI(model="gpt-4.1-mini"))

    evaluation_dataset = EvaluationDataset.from_pandas(dataset.to_pandas())
    custom_run_config = RunConfig(timeout=360)

    result = evaluate(
        dataset=evaluation_dataset,
        metrics=[],
        llm=evaluator_llm,
        run_config=custom_run_config
    )
    metric_map[retriever_name] = result
    dataset = copy_dataset
        
